In [1]:
import numpy as np
import random

class TicTacToe:
    def __init__(self):
        self.board = [''] * 9
        self.current_winner = None

    def reset(self):
        self.board = [''] * 9
        self.current_winner = None
        return self.get_state()

    def get_state(self):
        return "".join(self.board)

    def available_actions(self):
        return [i for i, spot in enumerate(self.board) if spot == '']

    def make_move(self, square, letter):
        if self.board[square] == '':
            self.board[square] = letter
            if self.winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def winner(self, square, letter):
        row_ind = square // 3
        if all([self.board[row_ind*3+i] == letter for i in range(3)]):
            return True
            
        col_ind = square % 3
        if all([self.board[col_ind + 3*i] == letter for i in range(3)]):
            return True
            
        if square % 2 == 0:
            if all([self.board[i] == letter for i in [0, 4, 8]]) or all([self.board[i] == letter for i in [2, 4, 6]]):
                return True
        return False

    def is_done(self):
        return self.current_winner is not None or '' not in self.board

    def render(self):
        print()
        for i in range(3):
            print(self.board[i*3:(i+1)*3])
        print()

In [2]:
class QLearningAgent:
    def __init__(self, alpha=0.3, gamma=0.9, epsilon=0.2):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, available_actions):
        # Epsilon-greedy policy
        if random.random() < self.epsilon:
            return random.choice(available_actions)
        qs = [self.get_q(state, a) for a in available_actions]
        max_q = max(qs)
        max_actions = [a for a, q in zip(available_actions, qs) if q == max_q]
        return random.choice(max_actions)

    def learn(self, state, action, reward, next_state, next_available_actions, done):
        current_q = self.get_q(state, action)
        if done:
            target = reward
        else:
            next_qs = [self.get_q(next_state, a) for a in next_available_actions]
            target = reward + self.gamma * max(next_qs)
        self.q_table[(state, action)] = current_q + self.alpha * (target - current_q)

In [3]:
def train(agent, episodes=50000):
    for episode in range(episodes):
        game = TicTacToe()
        state = game.reset()
        done = False
        while not done:
            available = game.available_actions()
            action = agent.choose_action(state, available)
            game.make_move(action, 'X')
            
            if game.current_winner == 'X':
                agent.learn(state, action, reward=1, next_state=None, next_available_actions=[], done=True)
                break
            elif game.is_done():
                agent.learn(state, action, reward=0.5, next_state=None, next_available_actions=[], done=True)
                break
                
            opp_available = game.available_actions()
            opp_action = random.choice(opp_available)
            game.make_move(opp_action, 'O')
            next_state = game.get_state()
            next_available = game.available_actions()
            
            if game.current_winner == 'O':
                agent.learn(state, action, reward=-1, next_state=next_state, next_available_actions=next_available, done=True)
                break
            elif game.is_done():
                agent.learn(state, action, reward=0.5, next_state=next_state, next_available_actions=next_available, done=True)
                break
            else:
                agent.learn(state, action, reward=0, next_state=next_state, next_available_actions=next_available, done=False)
                
            state = next_state

In [4]:
def play(agent):
    game = TicTacToe()
    state = game.reset()
    done = False
    game.render()
    while not done:
        available = game.available_actions()
        action = agent.choose_action(state, available)
        game.make_move(action, 'X')  # X represents the agent's move
        print("Agent's Move:")
        game.render()
        
        if game.current_winner == 'X':
            print("Agent Wins!")
            break
        elif game.is_done():
            print("Draw!")
            break
            
        opp_available = game.available_actions()
        opp_action = int(input(f"Your move {opp_available}: "))
        while opp_action not in opp_available:
            opp_action = int(input(f"Invalid! Choose from {opp_available}: "))
        game.make_move(opp_action, 'O')
        game.render()
        
        if game.current_winner == 'O':
            print("You Win!")
            break
        state = game.get_state()

In [5]:
agent = QLearningAgent(alpha=0.3, gamma=0.9, epsilon=0.2)
train(agent, episodes=50000)
play(agent)


['', '', '']
['', '', '']
['', '', '']

Agent's Move:

['', '', '']
['', '', 'X']
['', '', '']



Your move [0, 1, 2, 3, 4, 6, 7, 8]:  4



['', '', '']
['', 'O', 'X']
['', '', '']

Agent's Move:

['', '', '']
['', 'O', 'X']
['', 'X', '']



Your move [0, 1, 2, 3, 6, 8]:  8



['', '', '']
['', 'O', 'X']
['', 'X', 'O']

Agent's Move:

['X', '', '']
['', 'O', 'X']
['', 'X', 'O']



Your move [1, 2, 3, 6]:  6



['X', '', '']
['', 'O', 'X']
['O', 'X', 'O']

Agent's Move:

['X', '', 'X']
['', 'O', 'X']
['O', 'X', 'O']



Your move [1, 3]:  3



['X', '', 'X']
['O', 'O', 'X']
['O', 'X', 'O']

Agent's Move:

['X', 'X', 'X']
['O', 'O', 'X']
['O', 'X', 'O']

Agent Wins!
